# Physics-Informed Neural Network for multi-species forest dynamics

This notebook presents a proof-of-concept Physics-Informed Neural Network (PINN) for modelling the spatio-temporal dynamics of three interacting species in a two-dimensional domain.

The model combines a reaction-diffusion formulation with light competition represented through a Beer-Lambert attenuation term. In addition to reconstructing the species distributions $u_i(x,y,t)$, the PINN is used to solve an inverse problem by estimating the diffusion coefficients $D_i$ and intrinsic growth rates $r_i$.



### Reaction Diffusion PDE  

For species $i$ : 
$$
\frac{\partial u_i}{\partial t}
=
D_i\Delta u_i
+
r_i u_i
(1-u_i)
\left(1-\sum_{j=1}^{n}u_j\right)
\exp\left(-\sum_{j=1}^{n}\alpha_j u_j\right)
$$



## 2 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from scipy.interpolate import RegularGridInterpolator

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1 Parameters 

$$
(x,y)\in[0,10]\times[0,10]
$$

et :

$$
t\in[0,100]
$$ 

In [ ]:
Lx = 20.0
Ly = 20.0
T = 100.0

D_true = np.array([0.015, 0.010, 0.020], dtype=np.float32)
r_true = np.array([0.060, 0.050, 0.080], dtype=np.float32)



alpha_light = torch.tensor(
    [0.8, 1.0, 1.2],
    dtype=torch.float32,
    device=device
)
A = np.array([0.75, 0.65, 0.80], dtype=np.float32)
mu = np.array([
    [5.0, 5.0],    # espèce 1
    [15.0, 6.0],   # espèce 2
    [10.0, 15.0]   # espèce 3
], dtype=np.float32)
sigma = np.array([0.8, 1.0, 0.9], dtype=np.float32)


## 2  2D Initial Conditions

We initialize the populations with 2D Gaussian distributions:

$$
u_i(x,y,0)
=
A_i
\exp
\left(
-\frac{
(x-\mu_{x,i})^2+(y-\mu_{y,i})^2
}{
2\sigma_i^2
}
\right)
$$




In [ ]:
def initial_conditions(X, Y):
        u1 = A[0] * np.exp(
        -((X - mu[0,0])**2 + (Y - mu[0,1])**2) / (2 * sigma[0]**2)
    )
        u2 = A[1] * np.exp(
        -((X - mu[1,0])**2 + (Y - mu[1,1])**2) / (2 * sigma[1]**2)
    )
        u3 = A[2] * np.exp(
        -((X - mu[2,0])**2 + (Y - mu[2,1])**2) / (2 * sigma[2]**2)
    )

        return np.stack([u1, u2, u3], axis=-1)


## 3 Reference Finite Difference Solver

The 2D Laplacian is approximated by:

$$
\Delta u_{i,j}
\approx
\frac{
u_{i+1,j}-2u_{i,j}+u_{i-1,j}
}{\Delta x^2}
+
\frac{
u_{i,j+1}-2u_{i,j}+u_{i,j-1}
}{\Delta y^2}
$$

For an explicit scheme, we find that approximately:

$$
D\Delta t
\left(
\frac{1}{\Delta x^2}
+
\frac{1}{\Delta y^2}
\right)
\lesssim
\frac{1}{2}
$$




In [ ]:
Nx = 80
Ny = 80
Nt = 1001

x_grid = np.linspace(0, Lx, Nx)
y_grid = np.linspace(0, Ly, Ny)
t_grid = np.linspace(0, T, Nt)

dx = x_grid[1] - x_grid[0]
dy = y_grid[1] - y_grid[0]
dt = t_grid[1] - t_grid[0]

X_grid, Y_grid = np.meshgrid(x_grid, y_grid)

stability = 1/2
print("Stability ratios:", stability)


In [ ]:
def simulate_reference():

    U = np.zeros((Nt, Ny, Nx, 3), dtype=np.float32)
    U[0] = initial_conditions(X_grid, Y_grid)

    for n in range(Nt - 1):

        current = U[n]

        lap = np.zeros_like(current)

        lap[1:-1, 1:-1, :] = (
            (
                current[1:-1, 2:, :]
                - 2 * current[1:-1, 1:-1, :]
                + current[1:-1, :-2, :]
            ) / dx**2
            +
            (
                current[2:, 1:-1, :]
                - 2 * current[1:-1, 1:-1, :]
                + current[:-2, 1:-1, :]
            ) / dy**2
        )

        u1 = current[:, :, 0]
        u2 = current[:, :, 1]
        u3 = current[:, :, 2]

        #beer-lambert ------------------------------------------------

        u_total = u1 + u2 + u3

        light = np.exp(
            -(
                alpha_light[0].detach().cpu().numpy() * u1
                + alpha_light[1].detach().cpu().numpy() * u2
                + alpha_light[2].detach().cpu().numpy() * u3
            )
        )

        reaction1 = (
            r_true[0]
            * u1
            * (1 - u1)
            * (1 - u_total)
            * light
        )

        reaction2 = (
            r_true[1]
            * u2
            * (1 - u2)
            * (1 - u_total)
            * light
        )

        reaction3 = (
            r_true[2]
            * u3
            * (1 - u3)
            * (1 - u_total)
            * light
        )

        reaction = np.stack(
            [reaction1, reaction2, reaction3],
            axis=-1
        )

        U[n+1] = current + dt * (
            lap * D_true.reshape(1, 1, 3)
            + reaction
        )

        # Conditions de Neumann : flux nul
        U[n+1, 0, :, :] = U[n+1, 1, :, :]
        U[n+1, -1, :, :] = U[n+1, -2, :, :]
        U[n+1, :, 0, :] = U[n+1, :, 1, :]
        U[n+1, :, -1, :] = U[n+1, :, -2, :]

        # Évite les valeurs négatives
        U[n+1] = np.maximum(U[n+1], 0)

    return U


U_reference = simulate_reference()

print(U_reference.shape)

### Visualization

In [ ]:
times_to_plot = [0, 10, 50, 100]

for target_time in times_to_plot:
    idx = np.argmin(np.abs(t_grid - target_time))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for species in range(3):
        im = axes[species].imshow(
            U_reference[idx, :, :, species],
            extent=[0, Lx, 0, Ly],
            origin="lower",
            aspect="equal",
            cmap="summer"
        )
        axes[species].set_title(f"Specie {species+1} - t={t_grid[idx]:.1f}")
        axes[species].set_xlabel("x")
        axes[species].set_ylabel("y")
        plt.colorbar(im, ax=axes[species] )

    plt.tight_layout()
    plt.show()

## 4 Generate observations from reference




In [ ]:
N_obs = 500
noise_std = 0.02

x_obs = np.random.uniform(0 , Lx , N_obs)
y_obs = np.random.uniform(0 , Ly , N_obs)
t_obs = np.random.uniform(0 , T , N_obs)

points_obs = points_obs = np.column_stack([t_obs, y_obs, x_obs])

observations = np.zeros((N_obs, 3), dtype=np.float32)

for species in range(3):
    interpolator = RegularGridInterpolator(
        (t_grid, y_grid, x_grid),
        U_reference[:, :, :, species]
    )
    observations[:, species] = interpolator(points_obs)

observations_noisy = observations + np.random.normal(
    0.0, noise_std, observations.shape
).astype(np.float32)

We normalize data and derivative : 

$$
\tilde{x}=2\frac{x}{L_x}-1
$$

$$
\tilde{y}=2\frac{y}{L_y}-1
$$

$$
\tilde{t}=2\frac{t}{T}-1
$$


$$
u_x=\frac{2}{L_x}u_{\tilde{x}}
$$

$$
u_y=\frac{2}{L_y}u_{\tilde{y}}
$$

$$
u_t=\frac{2}{T}u_{\tilde{t}}
$$

$$
u_{xx}=\left(\frac{2}{L_x}\right)^2u_{\tilde{x}\tilde{x}}
$$

$$
u_{yy}=\left(\frac{2}{L_y}\right)^2u_{\tilde{y}\tilde{y}}
$$


In [ ]:
def normalize_x(x):
    return 2 * (x/Lx) -1 

def normalize_y(y):
    return 2 * (y/Ly) -1 

def normalize_t(t):
    return 2 * (t/T) -1 


## PINN Architecture : 

- 3 hidden layers of 64 neurons with Tanh activation function


In [ ]:
class MultiSpeciesPINN2D(nn.Module):
    def __init__(self, hidden_dim=64, hidden_layers=4):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.Tanh()
        )

        self.hidden_layers = nn.ModuleList()

        for _ in range(hidden_layers - 1):
            self.hidden_layers.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.Tanh()
                )
            )

        self.output_layer = nn.Linear(hidden_dim, 3)

    def forward(self, x):

        x = self.input_layer(x)

        for layer in self.hidden_layers:
            x = layer(x)

        x = self.output_layer(x)

        return x
model = MultiSpeciesPINN2D().to(device)


### Parameters learned

We use a logarithm on the parameters to avoid negative values. 

In [ ]:
#D_true = np.array([0.015, 0.010, 0.020], dtype=np.float32)
#r_true = np.array([0.060, 0.050, 0.080], dtype=np.float32)

D_initial_guess = torch.tensor([0.012, 0.008, 0.014], device=device)
r_initial_guess = torch.tensor([0.050, 0.040, 0.070], device=device)

rho_D = nn.Parameter(torch.log(D_initial_guess.clone()))
rho_r = nn.Parameter(torch.log(r_initial_guess.clone()))




## PDE Loss



In [ ]:
def compute_derivatives(model, X):
    X = X.clone().detach().requires_grad_(True)

    U = model(X)

    Ut_list = []
    Lap_list = []

    for i in range(3):
        u = U[:, i:i+1]

        grad_u = torch.autograd.grad(inputs=X , outputs=u , grad_outputs=torch.ones_like(u) , create_graph=True)[0]

        u_x = grad_u[:,0:1]
        u_y = grad_u[:,1:2]
        u_t = grad_u[:,2:3]

        grad_ux = torch.autograd.grad(inputs=X , outputs=u_x , grad_outputs=torch.ones_like(u_x) , create_graph=True)[0]
        grad_uy = torch.autograd.grad(inputs=X , outputs=u_y , grad_outputs=torch.ones_like(u_y) , create_graph=True)[0]

        u_xx = grad_ux[:,0:1]
        u_yy = grad_uy[:,1:2]

        u_t = (2.0 / T) * u_t
        u_xx = (2.0 / Lx)**2 * u_xx
        u_yy = (2.0 / Ly)**2 * u_yy

        Ut_list.append(u_t)
        Lap_list.append(u_xx + u_yy)

    return U, torch.cat(Ut_list, dim=1), torch.cat(Lap_list, dim=1)


In [ ]:

def pde_residuals(model, X_f , D ,r ):
    U, U_t, Lap = compute_derivatives(model, X_f)

    u1 = U[:, 0:1]
    u2 = U[:, 1:2]
    u3 = U[:, 2:3]

    u_total = u1 + u2 + u3

    
    light = torch.exp(
    -(
        alpha_light[0] * u1
        + alpha_light[1] * u2
        + alpha_light[2] * u3
    )
)

    reaction1 = r[0] * u1 * (1 - u1) * (1 - u_total) * light
    reaction2 = r[1] * u2* (1 - u2) * (1 - u_total) * light
    reaction3 = r[2] * u3* (1 - u3) * (1 - u_total) * light

    f1 = U_t[: , 0:1] -  (D[0] * Lap[:, 0:1] - reaction1)
    f2 = U_t[: , 1:2] -  (D[1] * Lap[:, 1:2] - reaction2)

    f3 = U_t[: , 2:3] -  (D[2] * Lap[:, 2:3] - reaction3)

    return torch.cat([f1, f2, f3], dim=1)


In [ ]:
N_f = 8000

x_f = np.random.uniform(0, Lx, N_f)
y_f = np.random.uniform(0, Ly, N_f)
t_f = np.random.uniform(0, T, N_f)

X_f = torch.tensor(
    np.column_stack([
        normalize_x(x_f),
        normalize_y(y_f),
        normalize_t(t_f)
    ]),
    dtype=torch.float32,
    device=device
)

In [ ]:
def physics_loss(model, X_f , D , r):
    residuals = pde_residuals(model , X_f , D , r)
    return torch.mean(residuals ** 2)


## Data Loss  

In [ ]:
X_obs = torch.tensor(
    np.column_stack([
        normalize_x(x_obs),
        normalize_y(y_obs),
        normalize_t(t_obs)
    ]),
    dtype=torch.float32,
    device=device
)

U_obs = torch.tensor(
    observations_noisy,
    dtype=torch.float32,
    
    device=device
)

def data_loss(model):
    prediction = model(X_obs)
    return torch.mean((prediction - U_obs)**2)

## Initial Loss 

Generate grid for inital points of $30 \times 30$

In [ ]:
N_ic_side = 30

x_ic = np.linspace(0, Lx, N_ic_side)
y_ic = np.linspace(0, Ly, N_ic_side)

X_ic_grid, Y_ic_grid = np.meshgrid(x_ic, y_ic)

x_i = X_ic_grid.ravel()
y_i = Y_ic_grid.ravel()
t_i = np.zeros_like(x_i)

X_i = torch.tensor(
    np.column_stack([
        normalize_x(x_i),
        normalize_y(y_i),
        normalize_t(t_i)
    ]),
    dtype=torch.float32,
    device=device
)

U_i = torch.tensor(
    initial_conditions(X_ic_grid, Y_ic_grid).reshape(-1, 3),
    dtype=torch.float32,
    device=device
)

def initial_loss(model):
    prediction = model(X_i)
    return torch.mean((prediction - U_i)**2)

### Boundary Loss

In [ ]:
N_b = 300

# Temps aléatoires communs aux points de frontière
t_b = np.random.uniform(0, T, N_b)

# -------------------------
# Bords gauche et droit
# -------------------------

y_lr = np.random.uniform(0, Ly, N_b)

X_left = torch.tensor(
    np.column_stack([
        normalize_x(np.zeros(N_b)),
        normalize_y(y_lr),
        normalize_t(t_b)
    ]),
    dtype=torch.float32,
    device=device
)

X_right = torch.tensor(
    np.column_stack([
        normalize_x(np.full(N_b, Lx)),
        normalize_y(y_lr),
        normalize_t(t_b)
    ]),
    dtype=torch.float32,
    device=device
)

# -------------------------
# Bords bas et haut
# -------------------------

x_bt = np.random.uniform(0, Lx, N_b)

X_bottom = torch.tensor(
    np.column_stack([
        normalize_x(x_bt),
        normalize_y(np.zeros(N_b)),
        normalize_t(t_b)
    ]),
    dtype=torch.float32,
    device=device
)

X_top = torch.tensor(
    np.column_stack([
        normalize_x(x_bt),
        normalize_y(np.full(N_b, Ly)),
        normalize_t(t_b)
    ]),
    dtype=torch.float32,
    device=device
)

def normal_derivative_loss(model, X, spatial_index, scale):

    X = X.clone().detach().requires_grad_(True)

    U = model(X)

    derivatives = []

    for i in range(3):

        u_i = U[:, i:i+1]

        grad_u = torch.autograd.grad(
            u_i,
            X,
            grad_outputs=torch.ones_like(u_i),
            create_graph=True
        )[0]

        # dérivée par rapport à x_norm ou y_norm
        du_dn_norm = grad_u[:, spatial_index:spatial_index+1] # si on regarde par rapport à x ou y 

        # conversion vers la dérivée physique
        du_dn = scale * du_dn_norm

        derivatives.append(du_dn)

    derivatives = torch.cat(derivatives, dim=1)

    return torch.mean(derivatives**2)


In [ ]:
def boundary_loss(model):

    loss_left = normal_derivative_loss(
        model,
        X_left,
        spatial_index=0,
        scale=2.0 / Lx
    )

    loss_right = normal_derivative_loss(
        model,
        X_right,
        spatial_index=0,
        scale=2.0 / Lx
    )

    loss_bottom = normal_derivative_loss(
        model,
        X_bottom,
        spatial_index=1,
        scale=2.0 / Ly
    )

    loss_top = normal_derivative_loss(
        model,
        X_top,
        spatial_index=1,
        scale=2.0 / Ly
    )

    return (
        loss_left
        + loss_right
        + loss_bottom
        + loss_top
    ) / 4.0

## Total Loss 

$$
\mathcal{L}
=
\lambda_{data}\mathcal{L}_{data}
+
\lambda_{PDE}\mathcal{L}_{PDE}
+
\lambda_{IC}\mathcal{L}_{IC}
+
\lambda_{BC}\mathcal{L}_{BC}
$$


In [ ]:
lambda_data = 1.0
lambda_pde = 1.0
lambda_ic = 1.0
lambda_bc = 1.0

def total_loss(model , D , r , lambda_data = 1.0 , lambda_pde = 1.0 , lambda_ic = 1.0 , lambda_bc = 1.0):
    l_data = data_loss(model)
    l_pde = physics_loss(model, X_f , D  , r)
    l_ic = initial_loss(model)
    l_bc = boundary_loss(model)

    total = (
        lambda_data * l_data
        + lambda_pde * l_pde
        + lambda_ic * l_ic
        + lambda_bc * l_bc
    )

    return total, l_data, l_pde, l_ic, l_bc

## Training 

In [ ]:

trainable_params = list(model.parameters()) + [rho_D, rho_r]

D_fixed = torch.tensor(D_true, dtype=torch.float32, device=device)
r_fixed = torch.tensor(r_true, dtype=torch.float32, device=device)





### Pretraining PINN with fixed inverse parameters  

We pre-train the model without the $D_i,r_i$ terms to avoid a degenerate solution in the first few epochs 

In [ ]:


optimizer_pretrain = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

epochs_pretrain = 4000

history = {
    "total": [],
    "data": [],
    "pde": [],
    "ic": [],
    "bc": [],
    "D": [],
    "r": []
}

for epoch in range(epochs_pretrain):

    optimizer_pretrain.zero_grad()

    loss, l_data, l_pde, l_ic, l_bc = total_loss(model , D_fixed ,r_fixed)

    loss.backward()

    optimizer_pretrain.step()

    history["total"].append(loss.item())
    history["data"].append(l_data.item())
    history["pde"].append(l_pde.item())
    history["ic"].append(l_ic.item())
    history["bc"].append(l_bc.item())

    history["D"].append(
        D_fixed.detach().cpu().numpy().copy()
    )

    history["r"].append(
        r_fixed.detach().cpu().numpy().copy()
    )

    if epoch % 500 == 0:
        print(
            f"Pretrain {epoch:5d} | "
            f"loss = {loss.item():.3e} | "
            f"D = {D_fixed.detach().cpu().numpy()} | "
            f"r = {r_fixed.detach().cpu().numpy()}"

    
        )



## Training with all paramers for inverse problem   

In [ ]:
optimizer = torch.optim.Adam([
    {
        "params": model.parameters(),
        "lr": 1e-4
    },
    {
        "params": [rho_D, rho_r],
        "lr": 1e-6
    }
])

epochs_inverse = 14000

for epoch in range(epochs_inverse):

    optimizer.zero_grad()

    D_now = torch.exp(rho_D)
    r_now = torch.exp(rho_r)

    loss, l_data, l_pde, l_ic, l_bc = total_loss(
        model,
        D_now,
        r_now
    )

    loss.backward()

    optimizer.step()

    # Recompute after optimizer.step() 
    D_now = torch.exp(rho_D)
    r_now = torch.exp(rho_r)

    history["total"].append(loss.item())
    history["data"].append(l_data.item())
    history["pde"].append(l_pde.item())
    history["ic"].append(l_ic.item())
    history["bc"].append(l_bc.item())

    history["D"].append(
        D_now.detach().cpu().numpy().copy()
    )

    history["r"].append(
        r_now.detach().cpu().numpy().copy()
    )

    if epoch % 500 == 0:
        print(
            f"Inverse {epoch:5d} | "
            f"loss = {loss.item():.3e} | "
            f"D = {D_now.detach().cpu().numpy()} | "
            f"r = {r_now.detach().cpu().numpy()}"
        )

### L-BGFS
L-BFGS is used as a final refinement step after Adam to improve convergence toward a local optimum.


In [ ]:
# =========================================================
# 2) L-BFGS
# =========================================================

optimizer_lbfgs = torch.optim.LBFGS(
    trainable_params,
    lr=1.0,
    max_iter=500,
    max_eval=625,
    tolerance_grad=1e-9,
    tolerance_change=1e-12,
    history_size=100,
    line_search_fn="strong_wolfe"
)

lbfgs_history = []

In [ ]:
def closure():

    optimizer_lbfgs.zero_grad()

    D_now = torch.exp(rho_D)
    r_now = torch.exp(rho_r)

    loss, l_data, l_pde, l_ic, l_bc = total_loss(
        model,
        D_now,
        r_now
    )

    loss.backward()

    lbfgs_history.append({
        "total": loss.item(),
        "data": l_data.item(),
        "pde": l_pde.item(),
        "ic": l_ic.item(),
        "bc": l_bc.item()
    })

    return loss

In [ ]:
optimizer_lbfgs.step(closure)

## Plot losses 

In [ ]:
D_learned = torch.exp(rho_D).detach().cpu().numpy()
r_learned = torch.exp(rho_r).detach().cpu().numpy()

D_error = np.abs(D_learned - D_true) / np.abs(D_true)
r_error = np.abs(r_learned - r_true) / np.abs(r_true)

print("True D:", D_true)
print("Estimated D:", D_learned)
print(" D error:", D_error)

print()
print("True r :", r_true)
print("Estimated r:", r_learned)
print(" r Error :", r_error)


loss_names = ["total", "data", "pde", "ic", "bc"]

loss_colors = {
    "total": "black",
    "data": "tab:blue",
    "pde": "tab:red",
    "ic": "tab:green",
    "bc": "tab:orange"
}

adam_end = len(history["total"])

plt.figure(figsize=(9, 6))

for name in loss_names:
    adam_values = history[name]
    lbfgs_values = [h[name] for h in lbfgs_history]

    all_values = np.concatenate([adam_values, lbfgs_values])

    plt.plot(
        all_values,
        color=loss_colors[name],
        linewidth=1.5,
        label=name
    )

plt.axvline(
    adam_end,
    linestyle="--",
    linewidth=2,
    color="gray",
    label="L-BFGS"
)

plt.yscale("log")
plt.xlabel("Optimization step")
plt.ylabel("Loss")
plt.title("Loss PINN 3 species (ADAM+L-BGFS)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Spatial Evaluation

Reconstruction at multiple time points relative to the reference grid 

for :

$$
t=0,\;10,\;50,\;100
$$




In [ ]:
times_to_plot = [0, 10, 50, 100]

for target_time in times_to_plot:
    idx = np.argmin(np.abs(t_grid - target_time))

    x_flat = X_grid.ravel()
    y_flat = Y_grid.ravel()
    t_flat = np.full_like(x_flat, target_time)

    X_eval = torch.tensor(
        np.column_stack([
            normalize_x(x_flat),
            normalize_y(y_flat),
            normalize_t(t_flat)
        ]),
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        pred = model(X_eval).cpu().numpy().reshape(Ny, Nx, 3)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    for species in range(3):
        im0 = axes[0, species].imshow(
            U_reference[idx, :, :, species],
            extent=[0, Lx, 0, Ly],
            origin="lower",
            cmap="summer"
        )
        axes[0, species].set_title(f"Specie reference {species+1}")
        plt.colorbar(im0, ax=axes[0, species])

        im1 = axes[1, species].imshow(
            pred[:, :, species],
            extent=[0, Lx, 0, Ly],
            origin="lower",
            cmap="summer"
        )
        axes[1, species].set_title(f"PINN Specie {species+1}")
        plt.colorbar(im1, ax=axes[1, species])

    fig.suptitle(f"t = {target_time}")
    plt.tight_layout()
    plt.show()

